# 02 - Legacy Function Calling (Deprecated)

This notebook demonstrates the **deprecated** `functions` format for educational purposes. While this format still works, it's been superseded by the modern `tools` format.

## ⚠️ Important Note
This format is **deprecated** as of November 2023. Use this notebook to:
- Understand legacy code you might encounter
- See the evolution of the API
- Learn why the modern format is better

**For new projects, skip to `03_function_calling_modern.ipynb`** 🚀

In [1]:
import os
import json
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

# Initialize client (same as basic setup)
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_OPENAI_GPT4O_DEPLOYMENT_NAME")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_GPT4O_API_VERSION")

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version=api_version,
)

print("✅ Legacy function calling setup complete!")

✅ Legacy function calling setup complete!


## 🏛️ Legacy Functions Format

The old format used:
- `functions` parameter (array of function definitions)
- `function_call` parameter (to control calling behavior)
- `function_call` response (single function call result)

In [2]:
# Define weather function using LEGACY format
legacy_functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "Temperature unit"
                }
            },
            "required": ["location"]
        }
    }
]

print("🏛️ Legacy function definition created:")
print(json.dumps(legacy_functions[0], indent=2))

🏛️ Legacy function definition created:
{
  "name": "get_current_weather",
  "description": "Get the current weather in a given location",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {
        "type": "string",
        "description": "The city and state, e.g. San Francisco, CA"
      },
      "unit": {
        "type": "string",
        "enum": [
          "celsius",
          "fahrenheit"
        ],
        "description": "Temperature unit"
      }
    },
    "required": [
      "location"
    ]
  }
}


In [3]:
# Make a call using LEGACY functions format
messages = [
    {
        "role": "user",
        "content": "What's the weather like in Amsterdam today?"
    }
]

# Legacy API call
legacy_completion = client.chat.completions.create(
    model=deployment,
    messages=messages,
    functions=legacy_functions,  # 🏛️ DEPRECATED: Use functions parameter
    function_call="auto",       # 🏛️ DEPRECATED: Use function_call parameter
    max_tokens=800,
    temperature=0.7
)

print("📞 LEGACY FUNCTION CALL RESULT:")
print("=" * 50)
message = legacy_completion.choices[0].message
print(f"Role: {message.role}")
print(f"Content: {message.content}")
print(f"Function Call: {message.function_call}")
print("=" * 50)

📞 LEGACY FUNCTION CALL RESULT:
Role: assistant
Content: None
Function Call: FunctionCall(arguments='{"location":"Amsterdam"}', name='get_current_weather')


## 🔍 Legacy Response Analysis

Notice the structure of the legacy response:

In [4]:
# Analyze the legacy response structure
if hasattr(message, 'function_call') and message.function_call:
    function_call = message.function_call
    print("🏛️ LEGACY FUNCTION CALL STRUCTURE:")
    print(f"   📞 Function Name: {function_call.name}")
    print(f"   📋 Arguments (raw): {function_call.arguments}")
    
    # Parse the arguments
    try:
        args = json.loads(function_call.arguments)
        print(f"   📋 Arguments (parsed): {args}")
        print(f"   📍 Location: {args.get('location')}")
        print(f"   🌡️ Unit: {args.get('unit', 'Not specified')}")
    except json.JSONDecodeError:
        print("   ❌ Failed to parse arguments as JSON")
    
    print("\n🏁 Finish Reason:", legacy_completion.choices[0].finish_reason)
else:
    print("❌ No function call detected")

🏛️ LEGACY FUNCTION CALL STRUCTURE:
   📞 Function Name: get_current_weather
   📋 Arguments (raw): {"location":"Amsterdam"}
   📋 Arguments (parsed): {'location': 'Amsterdam'}
   📍 Location: Amsterdam
   🌡️ Unit: Not specified

🏁 Finish Reason: function_call


## 🎭 Simulating Function Execution

Now let's complete the conversation by "executing" the function and providing results:

In [5]:
# Simulate the weather function execution
def get_current_weather(location, unit="celsius"):
    """Simulate a weather API call"""
    weather_data = {
        "location": location,
        "temperature": "18" if unit == "celsius" else "64",
        "unit": unit,
        "description": "Partly cloudy with light rain",
        "humidity": "78%",
        "wind_speed": "12 km/h"
    }
    return json.dumps(weather_data)

# Execute the function with the AI's arguments
if hasattr(message, 'function_call') and message.function_call:
    args = json.loads(message.function_call.arguments)
    function_result = get_current_weather(
        location=args["location"],
        unit=args.get("unit", "celsius")
    )
    
    print("🌤️ Function execution result:")
    print(function_result)
    
    # Add function result to conversation using LEGACY format
    messages.append({
        "role": "assistant",
        "content": None,
        "function_call": {
            "name": message.function_call.name,
            "arguments": message.function_call.arguments
        }
    })
    
    # Add function response using LEGACY format
    messages.append({
        "role": "function",                    # 🏛️ LEGACY: "function" role
        "name": message.function_call.name,    # 🏛️ LEGACY: function name
        "content": function_result             # 🏛️ LEGACY: function result
    })
    
    print("\n✅ Function result added to conversation")
else:
    print("❌ No function call to execute")

🌤️ Function execution result:
{"location": "Amsterdam", "temperature": "18", "unit": "celsius", "description": "Partly cloudy with light rain", "humidity": "78%", "wind_speed": "12 km/h"}

✅ Function result added to conversation


In [6]:
# Get the final response with function results
final_completion = client.chat.completions.create(
    model=deployment,
    messages=messages,
    functions=legacy_functions,  # Still provide function definitions
    max_tokens=800,
    temperature=0.7
)

print("🎯 FINAL AI RESPONSE (with weather data):")
print("=" * 60)
print(final_completion.choices[0].message.content)
print("=" * 60)

print("\n💰 TOKEN USAGE COMPARISON:")
print(f"   🥇 First call: {legacy_completion.usage.total_tokens} tokens")
print(f"   🥈 Second call: {final_completion.usage.total_tokens} tokens")
print(f"   🔢 Total: {legacy_completion.usage.total_tokens + final_completion.usage.total_tokens} tokens")

🎯 FINAL AI RESPONSE (with weather data):
The weather in Amsterdam today is partly cloudy with light rain. The temperature is 18°C, the humidity is 78%, and the wind speed is 12 km/h.

💰 TOKEN USAGE COMPARISON:
   🥇 First call: 99 tokens
   🥈 Second call: 189 tokens
   🔢 Total: 288 tokens


## ⚠️ Problems with Legacy Format

The legacy `functions` format has several limitations:

### 🚫 Single Function Limitation
- Can only call **ONE** function per turn
- No parallel function execution
- Inefficient for complex workflows

### 🔄 Multiple Round-Trips
- Requires separate API calls for each function
- Higher latency and token costs
- More complex conversation management

### 📝 Verbose Response Format
- Uses separate `function` role messages
- More complex message handling
- Harder to track in conversations

## 🚀 Why Modern Format is Better

The modern `tools` format addresses all these issues:
- ✅ **Multiple simultaneous tool calls**
- ✅ **Single API round-trip for multiple functions**
- ✅ **Cleaner response format with tool_call_id tracking**
- ✅ **Better performance and lower costs**

---

## 🎓 Key Takeaways

✅ **Legacy Understanding**: You now understand the deprecated format  
✅ **Limitations Identified**: Single function calls, multiple round-trips  
✅ **Evolution Context**: Why the API was improved  

## 🚀 Next Steps

Ready to see the modern, efficient approach?

**Continue to `03_function_calling_modern.ipynb`** to learn the current best practices! 🎯